# Example notebook

In [1]:
#reqs
import pandas as pd
import geopandas as gpd
import urllib
import json
import zzproxies
from zzproxies.core import registry
print("zzproxies version: ", zzproxies.__version__)

zzproxies version:  0.9.0


In [3]:
# helpers
def get_location_context(address: str, padding: float = 0.005):
    """
    Translates an address into a BBOX and an ISO Country Code.
    """
    url = f"https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(address)}&format=json&addressdetails=1&limit=1"
    headers = {"User-Agent": "zzproxies-research-lab"}
    
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req) as response:
        data = json.loads(response.read().decode())
        
    if not data:
        raise ValueError(f"Address '{address}' not found.")
        
    loc = data[0]
    lat, lon = float(loc["lat"]), float(loc["lon"])
    country_code = loc.get("address", {}).get("country_code", "").upper()
    
    # Generate BBOX [xmin, ymin, xmax, ymax]
    bbox = [lon - padding, lat - padding, lon + padding, lat + padding]
    
    return bbox, country_code

import lonboard
from lonboard import SolidPolygonLayer, Map
import palettable
import numpy as np

def plot_building_choropleth(gdf):
    # 1. Get unique building types and pick a palette
    unique_types = gdf['building_type'].unique().tolist()
    # Tableau_10 is great for distinct categorical data
    palette = palettable.tableau.Tableau_10.colors 
    
    # 2. Create a lookup dictionary { 'Residential': [r, g, b], ... }
    color_lookup = {
        b_type: palette[i % len(palette)] 
        for i, b_type in enumerate(unique_types)
    }
    
    # 3. Apply colors to the GDF (Lonboard needs a list of [R, G, B] for each row)
    # We use .map() for speed and cast to a list of lists
    gdf['color'] = gdf['building_type'].map(color_lookup)
    
    # 4. Handle any potential null colors (safety check)
    gdf['color'] = gdf['color'].apply(lambda x: x if x is not None else [200, 200, 200])

    # 5. Create the Layer
    layer = SolidPolygonLayer.from_geopandas(
        gdf,
        get_fill_color=np.array(gdf['color'].tolist(), dtype=np.uint8),
        opacity=0.8,
        pickable=True,
        auto_highlight=True
    )
    m = Map(layers=[layer])
    return m



In [9]:
# The research site
ADDRESS = "Tietäjäntie 1, Espoo, Finland" #"Stephansplatz, Vienna, Austria"

# Geocode the intent
bbox, country_code = get_location_context(ADDRESS, padding=0.005)

print(f"📍 Research Site: {ADDRESS}")
print(f"🌍 Country Code: {country_code}")
print(f"📦 BBOX: {bbox}")

📍 Research Site: Tietäjäntie 1, Espoo, Finland
🌍 Country Code: FI
📦 BBOX: [24.798048700000002, 60.1798973, 24.8080487, 60.189897300000005]


In [10]:
# Trigger the full pipeline

# Options: "upfront_gwp_supply", "activity_gwp_pcf", "slf_proxy"
PROXY_NAME = "upfront_gwp_supply" 

results = None
try:
    results = registry.execute_pipeline(
        proxy_name=PROXY_NAME,
        bbox=bbox,
        country_code=country_code
    )
    
    metadata = registry.get_metadata(PROXY_NAME)
    status = metadata["status"]
    
    print(f"Execution Successful!")
    print(f"Proxy: {status.name} (v{status.version})")
    print(f"Methodology: {status.description}")
    print(f"---")
    print(f"✅ Processed {len(results)} buildings in the district.")

except PermissionError as e:
    print(f"❌ Geographic Error: {e}")
except Exception as e:
    print(f"❌ Pipeline Error: {e}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Execution Successful!
Proxy: Supply-Side Embodied GWP (v1.0.0)
Methodology: Calculates upfront CO2e from building materials (A1-A3).
---
✅ Processed 197 buildings in the district.


In [11]:
#plot
df = pd.DataFrame(results)
gdf = gpd.GeoDataFrame(
        df, 
        geometry=gpd.GeoSeries.from_wkt(df['wkt']), 
        crs="EPSG:4326"
    )
m = plot_building_choropleth(gdf)
m

In [12]:
#
print(df['building_type'].value_counts())
df.head(3)


building_type
residential           88
apartment-condo       53
one-family-house      27
multi-family-house    14
commercial            12
public                 3
Name: count, dtype: int64


,id,building_type,num_floors,GFA,NFA_by_amenity,wkt,proxy_value,proxy_unit,GWPL
0,14954b66-957d-480e-948a-544f0b83d57d,apartment-condo,6,12530.0,"{'Retail & Food': 0.0, 'Accommodation': 0.0, '...","POLYGON ((24.8034658 60.1860024, 24.8031885 60...",9773.4,t_CO2e_upfront,L4: Critical Carbon Debt
1,7743b5fb-d41f-4c38-8c4e-5d71d2a9852c,apartment-condo,5,8470.0,"{'Retail & Food': 0.0, 'Accommodation': 0.0, '...","POLYGON ((24.8049351 60.183939, 24.8047277 60....",6606.6,t_CO2e_upfront,L4: Critical Carbon Debt
2,18cf2c71-6406-4a51-af4e-1b4c56dbb1a2,apartment-condo,4,6860.0,"{'Retail & Food': 0.0, 'Accommodation': 0.0, '...","POLYGON ((24.8013811 60.1838335, 24.8013931 60...",5350.8,t_CO2e_upfront,L4: Critical Carbon Debt


### Comparisons

In [4]:
# -- comparison ---
from zzproxies import UrbanImpactReport, compare_reports

bbox_a, country_code = get_location_context("Tikkurila, Vantaa", padding=0.003)
bbox_b, country_code = get_location_context("Malmi, Helsinki", padding=0.003)

# 1. Pipeline Execution
res_site_a = registry.execute_pipeline("upfront_gwp_supply", bbox_a, "FI")
res_site_b = registry.execute_pipeline("upfront_gwp_supply", bbox_b, "FI")

# 2. Report Creation
report_a = UrbanImpactReport("Tikkurila", res_site_a)
report_b = UrbanImpactReport("Malmi", res_site_b)

# 3. Comparison as DataFrame
comparison_table = pd.DataFrame(compare_reports([report_a, report_b]))

print("Impact-Verified Zoning Comparison ready!")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Impact-Verified Zoning Comparison ready!


In [5]:
comparison_table

,district_name,total_impact,unit,gwpl_distribution,critical_assets_L4,high_risk_assets_L3,primary_methodology
0,Tikkurila,336980.0,t_CO2e_upfront,"{'L4': 48, 'L3': 4, 'L2': 6, 'L1': 3, 'L0': 5}",48,4,N/A
1,Malmi,191970.0,t_CO2e_upfront,"{'L4': 45, 'L2': 4, 'L0': 3, 'L3': 1, 'L1': 2}",45,1,N/A
